[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/history/h01_difference_engine.ipynb)

# Turn the handle: Babbage's difference engine

In the 1820s Charles Babbage designed a machine that made tables of numbers by adding alone. Take the gaps between
a table's numbers, then the gaps between those gaps, and for any polynomial one column of gaps stops changing. A
machine that only adds can then print the whole table. In 1843 Ada Lovelace published a program for his later
machine, the Analytical Engine: Note G, for the Bernoulli numbers.

**How to use this notebook.** The grey boxes are cells. Click the first one, then press **Shift + Enter** (or the
▶ Run button) to run it. Run each cell in turn, from the top to the bottom. When a slider appears, drag it and watch.
Change any number you like and run the cell again. Nothing can break: if something looks odd, run everything again
from the top. It works in your browser (no install) and in Google Colab.

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look, and the engine itself. Run this cell first.
import math
from fractions import Fraction
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, IntText, ToggleButton

SLATE, SLATE2 = "#1f2b28", "#2a3a36"        # the slate ground
CHALK, DIM = "#ece6d6", "#93a097"           # chalk, and chalk smudge
CORAL = "#e3896b"                           # the one key thing in each figure
TEAL, SAGE, LILAC = "#6fb6bb", "#a0c27c", "#c9a0d3"

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#3b4c47", "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "figure.dpi": 100,
})

def turn(columns):
    """One turn of the handle: each column adds the column to its right (the old value), left to right."""
    cols = list(columns)
    for i in range(len(cols) - 1):
        cols[i] += cols[i + 1]
    return cols

def run(columns, turns):
    """Every state of the columns, from the start through `turns` turns."""
    out = [list(columns)]
    for _ in range(turns):
        out.append(turn(out[-1]))
    return out

def is_prime(m):
    return m > 1 and all(m % p for p in range(2, int(m ** 0.5) + 1))

def show(states, names, mark=None, last=None):
    """The columns as a table: one row per turn of the handle (only the last `last` turns, if given)."""
    first = max(0, len(states) - last) if last else 0
    print("turn  " + "".join(f"{n:>14}" for n in names))
    for k, s in enumerate(states[first:], start=first):
        flag = "   ← " + mark(s[0]) if mark and mark(s[0]) else ""
        print(f"{k:>4}  " + "".join(f"{v:>14,}" for v in s) + flag)

print("Ready. Run the next cell.")

## 1 · Squares, by adding alone

Three columns: the square, the gap to the next square, and the gap between gaps (always 2). Type the starting
numbers, then drag the handle. Start from 1, 3, 2 and every square comes out.

In [ ]:
def squares(value=1, gap=3, gap_of_gaps=2, turns=8):
    states = run([value, gap, gap_of_gaps], turns)
    show(states, ["value", "gap", "gap of gaps"])
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(range(turns + 1), [s[0] for s in states], "o-", color=CORAL, ms=7)
    ax.set_xlabel("turns of the handle"); ax.set_ylabel("value")
    ax.set_title("each turn adds; nothing is multiplied"); plt.show()

interact(squares, value=IntText(value=1, description="value"), gap=IntText(value=3, description="gap"),
         gap_of_gaps=IntText(value=2, description="gap of gaps"),
         turns=IntSlider(value=8, min=0, max=30, step=1, description="handle"));

## 2 · The engine that printed primes: n² + n + 41

Babbage showed his small engine off with n² + n + 41. Its columns start at 41, 2, 2. Every number it prints is
prime… for a while. Drag the handle and catch the first one that isn't.

In [ ]:
def primes(turns=10):
    states = run([41, 2, 2], turns)
    show(states, ["n² + n + 41", "gap", "gap of gaps"], last=8,
         mark=lambda v: "" if is_prime(v) else f"not prime: {v:,} = {int(v ** 0.5)} × {int(v ** 0.5)}" if int(v ** 0.5) ** 2 == v else "not prime")
    vals = [s[0] for s in states]
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.scatter(range(len(vals)), vals, color=[SAGE if is_prime(v) else CORAL for v in vals], s=40)
    ax.set_xlabel("n"); ax.set_ylabel("n² + n + 41"); ax.set_title("green: prime · coral: not prime"); plt.show()

interact(primes, turns=IntSlider(value=10, min=0, max=45, step=1, description="handle"));

**Notice.** For n = 0 to 39 every number is prime: forty in a row. At n = 40 the engine prints 1,681, which is
41 × 41. The machine never knew it was making primes; it was only following the rule.

## 3 · Any polynomial settles

Type a few numbers of a table and watch the gaps. For n³ (1, 8, 27, 64, 125, 216) the third gaps are all 6.
For a polynomial of degree d, the d-th gaps never change.

In [ ]:
def gaps(row):
    return [b - a for a, b in zip(row, row[1:])]

def gap_table(power=3, start=1, count=7):
    row = [n ** power for n in range(start, start + count)]
    k = 0
    while row:
        print(("value " if k == 0 else f"gaps {k}").ljust(8) + "  ".join(f"{v:,}" for v in row))
        if len(set(row)) == 1 and k > 0:
            print(f"\nsettles after {k} rounds of gaps"); break
        row = gaps(row); k += 1

interact(gap_table, power=IntSlider(value=3, min=1, max=7, step=1, description="power"),
         start=IntSlider(value=1, min=0, max=20, step=1, description="start n"),
         count=IntSlider(value=9, min=3, max=12, step=1, description="numbers"));

## 4 · The challenge from the video

A table begins 1, 5, 14, 30, 55. Use the method of differences to find the next number. Try it on paper first,
then press the button.

In [ ]:
def challenge(show_answer=False):
    row = [1, 5, 14, 30, 55]
    if not show_answer:
        print("1, 5, 14, 30, 55, ?   (press the button when you're ready)"); return
    g1, g2, g3 = gaps(row), gaps(gaps(row)), gaps(gaps(gaps(row)))
    print("gaps:", g1, " gaps of gaps:", g2, " and again:", g3)
    nxt = turn([row[-1], g1[-1] + g2[-1] + g3[-1], g2[-1] + g3[-1], g3[-1]])[0]
    print("the next number:", nxt, "   (these are 1², 1² + 2², 1² + 2² + 3², …)")

interact(challenge, show_answer=ToggleButton(value=False, description="show the answer"));

## 5 · The numbers in Note G

Lovelace's Note G computes a Bernoulli number on the Analytical Engine. They are a strange list of fractions that
Jacob Bernoulli found while adding up powers of whole numbers. Here they are, from the standard recurrence.

In [ ]:
def bernoulli(n):
    B = [Fraction(1)]
    for m in range(1, n + 1):
        B.append(-sum(math.comb(m + 1, k) * B[k] for k in range(m)) / (m + 1))
    return B

for k, b in enumerate(bernoulli(12)):
    if b != 0: print(f"B{k} = {b}")
print("\nNote G computes B8 = −1/30 in today's numbering (Lovelace called it B7).")

## Sources
MacTutor History of Mathematics: Charles Babbage; Ada Lovelace. Computer History Museum, "The Engines" and "A Modern
Sequel". L. F. Menabrea, *Sketch of the Analytical Engine*, with notes by A. A. L. (Taylor's Scientific Memoirs 3,
1843). All retrieved 2026-10-25.